<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; text-align: center; margin-bottom: 10px;">
  <div style="font-size: 64px; margin-bottom: 10px;">🎵</div>
  <h1 style="color: #e94560; font-size: 2.2em; margin: 0 0 8px 0; font-family: Georgia, serif;">Investigación de datos</h1>
  <h2 style="color: #a8dadc; font-size: 1.4em; margin: 0 0 20px 0; font-weight: normal;">Caso Chinook · Tienda de música digital</h2>
  <p style="color: #ccc; font-size: 0.95em; max-width: 500px; margin: 0 auto;">SQL + Python · Análisis exploratorio · Visualización</p>
</div>

## ¿Qué es Chinook?

**Chinook** es una base de datos de ejemplo de una tienda de música digital.
Incluye tablas conectadas entre sí como:

- `Customer` (clientes)
- `Invoice` (facturas)
- `InvoiceLine` (detalle de cada factura)
- `Track` (canciones)
- `Genre` (géneros musicales)

🎯 En este cuaderno vais a trabajar como analistas: formular preguntas, consultar datos y justificar conclusiones con evidencia.

---


# 🔍 Investigación: ¿quién mueve más dinero en Chinook?

Ya sabéis hacer consultas en DB Browser. Ahora el reto es conectar la base de datos desde Python y responder **4 preguntas de negocio** usando `pandas` + `sqlite3`.

No hay una solución única. Si tu consulta funciona y el resultado tiene sentido: está bien.

---

**Normas del reto:**
- Una celda por pregunta con tu consulta SQL dentro de `pd.read_sql(...)`
- El resultado visible (que se vea la tabla)
- **Una línea de texto** debajo explicando qué significa lo que ves
- En cada pregunta, haz primero una versión de prueba (por ejemplo con `LIMIT 5`) y luego la versión final

Al cerrar, revisaremos las soluciones **en conjunto en formato fórum** 👇

## Paso 0 · Conexión a la base de datos

Ejecuta esta celda primero. Si ves una tabla con nombres de tablas, estás dentro ✅

In [30]:
import sqlite3
import pandas as pd

# Ajusta la ruta si tu Chinook.db está en otra carpeta
con = sqlite3.connect("Chinook.db")

# ¿Qué tablas hay disponibles?
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


---

## Pregunta 0 (calentamiento) — ¿Cuántas facturas hay en total?

Antes de empezar las 4 preguntas, valida que todo funciona con una consulta muy corta.

Si esta sale bien, ya tienes conexión y sintaxis controladas ✅

In [31]:
# Pregunta 0: total de facturas
pd.read_sql("""
    SELECT COUNT(*) AS total_facturas
    FROM Invoice
""", con)

,total_facturas
0,412


---

## Pregunta 1 — ¿Qué empleado tiene más clientes asignados?

> 💡 Pista: hay una columna en la tabla `Customer` que apunta a la tabla `Employee`. ¿Cuál es?

Escribe tu consulta en la celda de abajo y muestra el resultado.

Cuando lo tengas, añade una línea en texto explicando qué significa el número que ves.

In [32]:
# Pregunta 1: ¿Qué empleado tiene más clientes asignados?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT SupportRepId AS empleados, 
    COUNT(CustomerId) AS Total_clientes
    FROM Customer
    GROUP BY empleados;
    """, con)

,empleados,Total_clientes
0,3,21
1,4,20
2,5,18


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 2 — ¿En qué mes del año se factura más? ¿Y menos?

> 💡 Pista: la fecha de factura está guardada como texto. Puedes extraer el mes con `strftime('%m', InvoiceDate)`.

Cuando lo tengas, añade una línea explicando si el patrón tiene sentido o te parece sorprendente.

In [33]:
# Pregunta 2: ¿En qué mes se factura más? ¿Y menos?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT *, strftime('%m', InvoiceDate) , SUM(Total)
    FROM Invoice
    GROUP BY InvoiceDate
    ORDER BY sum(total) DESC
    LIMIT 3;
            
""", con)

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,"strftime('%m', InvoiceDate)",SUM(Total)
0,404,6,2013-11-13 00:00:00,Rilská 3174/6,Prague,NaN,Czech Republic,14300,25.86,11,25.86
1,299,26,2012-08-05 00:00:00,2211 W Berry Street,Fort Worth,TX,USA,76110,23.86,08,23.86
2,194,46,2011-04-28 00:00:00,3 Chatham Street,Dublin,Dublin,Ireland,NaN,21.86,04,21.86


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 3 — ¿Hay clientes que hayan comprado solo una vez? ¿Cuántos son?

> 💡 Pista: una compra = una factura en la tabla `Invoice`. Agrupa por cliente y filtra los que solo tienen 1.

¿Sorprende la cantidad? Escribe tu conclusión debajo del resultado.

In [41]:
# Pregunta 3: ¿Cuántos clientes han comprado solo una vez?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT CustomerId, count(*) 
    FROM Invoice
    GROUP by CustomerId
    HAVING count(*) = 1;
    
""", con)

,CustomerId,count(*)


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 4 — ¿Cuál es el género musical más vendido por tracks comprados?

> 💡 Esta la haremos en 2 pasos:
> 1) Une `InvoiceLine` con `Track` para comprobar que puedes llegar a `TrackId` y `GenreId`.
> 2) Añade `Genre`, agrupa y ordena para obtener el top.

Esta es la más difícil. Si llegas aquí, enhorabuena 🎉

In [43]:
# Pregunta 4: ¿Cuál es el género más vendido por número de tracks comprados?

# Paso 1 (prueba): valida el cruce InvoiceLine + Track
pd.read_sql("""
    SELECT *
    FROM InvoiceLine il INNER JOIN Track t
    ON il.TrackId = t.TrackId
    LIMIT 5
""", con)

# Paso 2 (final): añade Genre, agrupa y ordena
pd.read_sql("""
    SELECT *, SUM(Quantity) 
    FROM InvoiceLine il INNER JOIN Track t
    ON il.TrackId = t.TrackId
    INNER JOIN Genre g
    ON g.GenreId = t.GenreId
    GROUP BY g.GenreId
    ORDER BY SUM(Quantity)
    LIMIT 5;
""", con)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice,GenreId,Name,SUM(Quantity)
0,24,5,117,0.99,1,117,Rock 'N' Roll Music,12,1,5,Chuck Berry,141923,2276788,0.99,5,Rock And Roll,6
1,469,88,2826,1.99,1,2826,Hero,227,3,18,NaN,2713755,506896959,1.99,18,Science Fiction,6
2,527,96,3214,1.99,1,3214,Phyllis's Wedding,251,3,22,NaN,1271521,258561054,1.99,22,Comedy,9
3,175,33,1036,0.99,1,1036,I Get A Kick Out Of You,83,1,12,cole porter,194429,6332441,0.99,12,Easy Listening,10
4,211,39,1250,0.99,1,1250,Gates Of Tomorrow,98,1,13,Bruce Dickinson/Janick Gers/Steve Harris,312032,12482688,0.99,13,Heavy Metal,12


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Cierre en fórum (sin entrega)

- No hay entrega individual hoy
- Guarda tus consultas para poder enseñarlas si te toca compartir
- Cerraremos la sesión resolviendo dudas y comparando enfoques **en conjunto**
- Piensa qué pregunta te costó más y por qué, para comentarla en el fórum